# 🌿 Train EfficientNet-B0 on PlantVillage for FarmerFriend

This notebook trains an **EfficientNet-B0** convolutional neural network on the **PlantVillage** dataset (~54k images across 38 classes).

### ⚡ Steps:
1. Ensure GPU is active: **Runtime** ➔ **Change runtime type** ➔ **T4 GPU**.
2. Run all cells in order.
3. At the end, the trained `plant_disease_effnet_b0.pth` will automatically download to your computer!
4. Place it in `c:\Users\HP\Desktop\inno\backend\models\plant_disease_effnet_b0.pth`.

### Step 1: Check GPU Availability

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies & Download PlantVillage Dataset

In [ ]:
!pip install kagglehub torchvision tqdm

import kagglehub
import os
import glob

print("[*] Downloading PlantVillage dataset from Kaggle...")
dataset_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("[*] Download complete! Path:", dataset_path)

# Find the folder containing the 38 class directories
candidates = glob.glob(f"{dataset_path}/**/color", recursive=True) + glob.glob(f"{dataset_path}/**/PlantVillage", recursive=True)
data_dir = candidates[0] if candidates else dataset_path

# If nested inside segmented/color
for root, dirs, files in os.walk(dataset_path):
    if len(dirs) == 38:
        data_dir = root
        break

print(f"[✓] Using Image Directory: {data_dir}")
print(f"[✓] Number of class folders found: {len(os.listdir(data_dir))}")

### Step 3: Setup PyTorch DataLoaders with Augmentation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# EfficientNet-B0 standard input resolution is 224x224
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(data_dir, transform=train_transform)
num_classes = len(full_dataset.classes)
print(f"Total images: {len(full_dataset)} across {num_classes} classes")

# 80% train / 20% validation
val_size = int(len(full_dataset) * 0.2)
train_size = len(full_dataset) - val_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

### Step 4: Build Pretrained EfficientNet-B0 Model

In [ ]:
# Load pretrained EfficientNet-B0 from torchvision
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Replace final classifier head for 38 PlantVillage classes
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(in_features, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler() # AMP for fast GPU training

### Step 5: Train for 8 Epochs (~10-15 minutes on T4 GPU)

In [ ]:
from tqdm import tqdm

EPOCHS = 8
best_val_acc = 0.0
save_filename = "plant_disease_effnet_b0.pth"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        total_train += labels.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.3f}"})

    scheduler.step()
    train_acc = (correct_train / total_train) * 100
    train_loss = running_loss / total_train

    # Validation
    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data).item()
            total_val += labels.size(0)

    val_acc = (correct_val / total_val) * 100
    print(f"\nEpoch {epoch:02d} Summary: Train Acc = {train_acc:.2f}% | Val Acc = {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_filename)
        print(f"⭐ New Best Model Saved to {save_filename} with Accuracy: {val_acc:.2f}%\n")

print(f"\n[✓] Finished Training! Highest Validation Accuracy: {best_val_acc:.2f}%")

### Step 6: Download the Weights File to Your Computer

In [ ]:
from google.colab import files
print("[*] Downloading plant_disease_effnet_b0.pth to your PC...")
files.download("plant_disease_effnet_b0.pth")
print("Move this file to: c:\\Users\\HP\\Desktop\\inno\\backend\\models\\plant_disease_effnet_b0.pth")